In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_34.pickle

In [ ]:
%%RecordEvent
%%time
### cell 35 ###

# Optimized pandas workflow for CPU & memory efficiency
# 1) grab_subset_of_data_47 only copies needed columns
# 2) combine_subset… loops over years instead of repeating calls
# 3) convert_df… uses vectorized division
# 4) framework consolidation uses a single pass and drops old columns in one go

def grab_subset_of_data_47(original_df, question_of_interest):
    subset_cols = [c for c in original_df.columns if question_of_interest in c]
    new_names   = [c.split('-')[-1].lstrip() for c in subset_cols]
    return (
        original_df.loc[:, subset_cols]
                   .rename(columns=dict(zip(subset_cols, new_names)))
                   .dropna(how='all', subset=new_names)
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest,
    include_2017=False
):
    # map each year to its raw DataFrame
    df_map = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10,
    }
    if include_2017:
        df_map['2017'] = responses_df_2017
    years = sorted(df_map)
    # subset and tag each year in one comprehension
    dfs = [
        grab_subset_of_data_47(df_map[y], question_of_interest).assign(year=y)
        for y in years
    ]
    df_combined = pd.concat(dfs)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_47(df, df_counts):
    df_pct = df_counts.copy()
    # total responses per year in same order as df_counts
    total = df['year'].value_counts().reindex(df_pct['year']).values
    cols  = df_pct.columns.difference(['year'])
    # vectorized percent conversion
    df_pct[cols] = df_pct[cols].mul(100).div(total, axis=0)
    return df_pct

# rename the 2018 columns once
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(
            question_of_interest_old_cell47,
            question_of_interest_new_cell47,
            regex=False
        )
)

# combine subsets across years
ml_df_combined_cell47, ml_df_combined_counts_cell47 = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)

# consolidate framework columns in one pass
ml_df2 = ml_df_combined_cell47.copy()
mapping = {
    'TensorFlow/Keras': {
        'cols': ['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
        'value': 'TensorFlow/Keras'
    },
    'PyTorch/Lightning/Fast.ai': {
        'cols': ['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
        'value': 'PyTorch/PyTorch Lightning/Fast.ai'
    },
    'Xgboost/LightGBM/CatBoost': {
        'cols': ['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
        'value': 'Xgboost/LightGBM/CatBoost'
    },
    'Scikit-learn': {
        'cols': ['Scikit—learn ', 'Scikit—learn ', 'learn ', 'Learn'],
        'value': 'Scikit-learn'
    }
}
drop_cols = []
for new_col, params in mapping.items():
    cols = params['cols']
    mask = ml_df2[cols].notna().any(axis=1)
    ml_df2[new_col] = np.where(mask, params['value'], np.nan)
    drop_cols.extend(cols)
ml_df2.drop(columns=drop_cols, inplace=True)

# regroup & percentages
ml_df_counts2 = ml_df2.groupby('year').count().reset_index()
ml_df_pct     = convert_df_of_counts_to_percentages_47(ml_df2, ml_df_counts2)

# select & reshape for plotting
ml_df_pct = ml_df_pct.loc[:, [
    'year',
    'Scikit-learn',
    'TensorFlow/Keras',
    'PyTorch/Lightning/Fast.ai',
    'Xgboost/LightGBM/CatBoost'
]]
df_cell47 = ml_df_pct.melt(
    id_vars=['year'],
    value_vars=[
        'Scikit-learn',
        'Xgboost/LightGBM/CatBoost',
        'TensorFlow/Keras',
        'PyTorch/Lightning/Fast.ai'
    ]
)
# preserve original post-melt behavior
df_cell47 = df.sort_values(by=['year', 'value'], ascending=True)
df_cell47 = df.rename(columns={'variable': ''})
df_cell47.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_35_try_0.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_35_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[35], f)


In [ ]:
opt_output = Out.get(4)